# GeoLegal-AI E1 baseline (Google Colab)
This notebook runs Tiny Aya without retrieval. It is a research experiment, not legal advice. All current answer keys are `PENDING_REVIEW`, so outputs are saved but accuracy is not scored.

In [ ]:
%pip install -q "transformers>=5.15,<6" "accelerate>=1.2,<2" safetensors


In [ ]:
# Store HF_TOKEN in Colab Secrets (key icon). Never paste it into the notebook.
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets and enable notebook access.')
login(token=hf_token, add_to_git_credential=False)
del hf_token

In [ ]:
from pathlib import Path
import json, sys, torch

REPO_ROOT = Path('/content/geolegal-ai-lexsi')
if not REPO_ROOT.exists():
    raise FileNotFoundError('Clone the repository to /content/geolegal-ai-lexsi first.')
sys.path.insert(0, str(REPO_ROOT))
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')

from src.dataset import load_scenarios
from src.baseline import load_model_once, generate_one, MODEL_ID, MODEL_REVISION
from src.evaluate import score_predictions, save_metrics

scenarios = load_scenarios(REPO_ROOT / 'data/eval/held_out_initial.jsonl')
print(f'Loaded {len(scenarios)} pending-review scenarios')
print(torch.cuda.get_device_name(0))

In [ ]:
# Load the tokenizer and model exactly once. Reuse these objects for every scenario.
tokenizer, model, torch_module = load_model_once(MODEL_ID, MODEL_REVISION)
print(MODEL_ID, MODEL_REVISION, model.device)

In [ ]:
predictions_path = REPO_ROOT / 'results/baseline_predictions.jsonl'
predictions_path.parent.mkdir(parents=True, exist_ok=True)
predictions = []
with predictions_path.open('w', encoding='utf-8') as handle:
    for index, scenario in enumerate(scenarios, 1):
        prediction = generate_one(tokenizer, model, torch_module, scenario)
        predictions.append(prediction)
        handle.write(json.dumps(prediction, ensure_ascii=False) + '\n')
        print(f'{index}/{len(scenarios)} {scenario.scenario_id}')
print(f'Saved {len(predictions)} outputs to {predictions_path}')

In [ ]:
# Pending answer keys are deliberately excluded from accuracy metrics.
metrics = score_predictions(scenarios, predictions)
metrics_path = REPO_ROOT / 'results/baseline_metrics.json'
save_metrics(metrics_path, metrics)
print(json.dumps(metrics, indent=2))
assert metrics['scored_count'] == 0
assert metrics['excluded_pending_review'] == 15